classifiy and stack

In [ ]:
import numpy as np
import pandas as pd
import cfg
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
import jieba
import seaborn as sns
import plotly.offline as py

py.init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.tools as tls
from collections import Counter
from scipy.misc import imread
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

df_all = pd.read_pickle(cfg.data_path + 'df_all_1k.pkl')
df_all.head()

In [ ]:
df_tr = df_all.loc[df_all['type'] == 'train']
df_te = df_all.loc[df_all['type'] == 'test']
print('df_tr.shape: ',df_tr.shape)

X = df_tr['query_seg_no_stopword']
y = df_tr['label']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

print(X_train[:5])
# examine the class distribution in y_train and y_test
print(y_train.value_counts(),'\n', y_test.value_counts())

In [ ]:
vect = CountVectorizer(lowercase=False, token_pattern=r'(?u)\b\w+\b|\,|\.|\;|\:')
vect

In [ ]:
# learn the vocabulary in the training data, then use it to create a document-term matrix
X_train_dtm = vect.fit_transform(X_train)

# transform the test data using the earlier fitted vocabulary, into a document-term matrix
X_test_dtm = vect.transform(X_test)
# examine the document-term matrix from X_test
X_train_dtm

In [ ]:
from sklearn.decomposition import TruncatedSVD

lsa = TruncatedSVD(n_components=4, n_iter=500)
lsa.fit(X_train_dtm)
assert isinstance(vect.get_feature_names, object)
terms = vect.get_feature_names()
for i, comp in enumerate(lsa.components_):
    termsInComp = zip(terms, comp)
    sortedterms = sorted(termsInComp, key=lambda x: x[1], reverse=True)[:10]
    print("Concept %d:" % i)
    for term in sortedterms:
        print(term[0])
    print(" ")

In [ ]:
from sklearn.naive_bayes import MultinomialNB
nb = MultinomialNB()

# tune hyperparameter alpha = [0.01, 0.1, 1, 10, 100]
from sklearn.model_selection import GridSearchCV
grid_values = {'alpha':[0.01, 0.1, 1.0, 10.0, 100.0]}
grid_nb = GridSearchCV(nb, param_grid=grid_values, scoring='neg_log_loss')
grid_nb.fit(X_train_dtm, y_train)
grid_nb.best_params_

In [ ]:
# set with recommended hyperparameters
nb = MultinomialNB(alpha=100.0)
# train the model using X_train_dtm & y_train
nb.fit(X_train_dtm, y_train)

# compute the accuracy of training data predictions
y_pred_train = nb.predict(X_train_dtm)
print('train acc:', metrics.accuracy_score(y_train, y_pred_train))

# make author (class) predictions for X_test_dtm
y_pred_test = nb.predict(X_test_dtm)
print('test acc:', metrics.accuracy_score(y_test, y_pred_test))

Add char count as a feature to martix


In [ ]:
def add_feature(X, feature_to_add):
    '''
    Returns sparse feature matrix with added feature.
    feature_to_add can also be a list of features.
    '''
    from scipy.sparse import csr_matrix, hstack
    return hstack([X, csr_matrix(feature_to_add).T], 'csr')

In [ ]:
from string import punctuation
print('punc:', punctuation)
X_train_chars = X_train.str.len()
X_train_punc = X_train.apply(lambda x: len([c for c in str(x) if c in punctuation]))
X_test_chars = X_test.str.len()
X_test_punc = X_test.apply(lambda x: len([c for c in str(x) if c in punctuation]))
print(X_train_punc[:5])

print(X_train_chars[:5])

X_train_dtm = add_feature(X_train_dtm, [X_train_chars, X_train_punc])
X_test_dtm = add_feature(X_test_dtm, [X_test_chars, X_test_punc])

X_train_dtm

In [ ]:
# set with recommended hyperparameters
nb = MultinomialNB(alpha=0.1)
# train the model using X_train_dtm & y_train
nb.fit(X_train_dtm, y_train)

In [ ]:
# compute the accuracy of training data predictions
y_pred_train = nb.predict(X_train_dtm)
print('train acc:', metrics.accuracy_score(y_train, y_pred_train))

# make author (class) predictions for X_test_dtm
y_pred_test = nb.predict(X_test_dtm)
print('test acc:', metrics.accuracy_score(y_test, y_pred_test))

# look at the confusion matrix for y_test
print(metrics.confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test))

In [ ]:
# calculate predicted probabilities for X_test_dtm
y_pred_prob = nb.predict_proba(X_test_dtm)
print(y_pred_prob[:5])

using Logistic Regression

In [ ]:
# import and instantiate the Logistic Regression model
from sklearn.linear_model import LogisticRegression
logreg = LogisticRegression(random_state=0)

In [ ]:
# tune hyperparameter
grid_values = {'C':[0.01, 0.1, 1.0, 3.0, 5.0]}
grid_logreg = GridSearchCV(logreg, param_grid=grid_values, scoring='neg_log_loss')
grid_logreg.fit(X_train_dtm, y_train)
grid_logreg.best_params_

In [ ]:
# set with recommended parameter
logreg = LogisticRegression(C=0.1, random_state=0)
# train the model using X_train_dtm & y_train
logreg.fit(X_train_dtm, y_train)

In [ ]:
# compute the accuracy of training data predictions
y_pred_train = logreg.predict(X_train_dtm)
print('train acc:', metrics.accuracy_score(y_train, y_pred_train))

# make author (class) predictions for X_test_dtm
y_pred_test = logreg.predict(X_test_dtm)
print('test acc:', metrics.accuracy_score(y_test, y_pred_test))

In [ ]:
# import and instantiate the XGB model
from xgboost import XGBClassifier
model = XGBClassifier()

In [ ]:
# tune hyperparameter
grid_values = {'max_depth': [3, 5, 7]}
grid_logreg = GridSearchCV(model, param_grid=grid_values, scoring='neg_log_loss')
grid_logreg.fit(X_train_dtm, y_train)
grid_logreg.best_params_

In [ ]:
# set with recommended parameter
model = XGBClassifier(max_depth=5)
# train the model using X_train_dtm & y_train
model.fit(X_train_dtm, y_train)

In [ ]:
# compute the accuracy of training data predictions
y_pred_train = model.predict(X_train_dtm)
print('train acc:', metrics.accuracy_score(y_train, y_pred_train))

# make author (class) predictions for X_test_dtm
y_pred_test = model.predict(X_test_dtm)
print('test acc:', metrics.accuracy_score(y_test, y_pred_test))

In [ ]:
import xgboost as xgb
### Plot the important variables ###
fig, ax = plt.subplots(figsize=(12,12))
xgb.plot_importance(model, max_num_features=50, height=0.8, ax=ax)
plt.show()

confusion matrix

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix

### From http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html#sphx-glr-auto-examples-model-selection-plot-confusion-matrix-py #
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        #print("Normalized confusion matrix")
    #else:
    #    print('Confusion matrix, without normalization')

    #print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
cnf_matrix = confusion_matrix(y_test, y_pred_test)
print(cnf_matrix)
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure(figsize=(8, 8))
plot_confusion_matrix(cnf_matrix, classes=[0, 1, 2, 3],
                      title='Confusion matrix of XGB, without normalization')
plt.show()